# ISIC 2024: TabM Decoupled Tabular Hyperparameter Tuning with Optuna

This notebook implements **Stage 1 (Tabular-Only Optimization)** of the decoupled hyperparameter tuning workflow for the ISIC 2024 Challenge.

### Why Tabular-Only Optimization?
1. **Extremely Fast Trials**: By skipping image loading and backbone forward passes, we can evaluate a single configuration in a fraction of the time.
2. **Mitigates Modality Collapse**: Tuning the tabular encoder parameters (PLE bins, embedding size, dropout, etc.) independently ensures the tabular model is fully expressive before combining it with the image backbone.
3. **Decoupled Architecture**: Directly maps tuned parameters back into the final `MidFusionLiteModel` staged training configuration.

In [ ]:
# =========================
# CELL 1: SETUP ENVIRONMENT
# =========================
import sys
from pathlib import Path

# Set project root path (edit as needed)
CODE_DATASET_ROOT = '/kaggle/input/mid-fusion-lite' 
if not Path(CODE_DATASET_ROOT).exists():
    CODE_DATASET_ROOT = '.'

if CODE_DATASET_ROOT not in sys.path:
    sys.path.insert(0, CODE_DATASET_ROOT)

# Offline / Online dependencies installation
offline_dir = Path('/kaggle/input/tabm-kaggle-wheels')
if offline_dir.exists():
    print('Installing dependencies offline...')
    !pip install /kaggle/input/tabm-kaggle-wheels/tabm-0.0.3-py3-none-any.whl /kaggle/input/tabm-kaggle-wheels/rtdl_num_embeddings-0.0.12-py3-none-any.whl --no-deps -q
else:
    print('Installing dependencies online...')
    !pip install -q tabm rtdl_num_embeddings optuna

print('Setup complete.')

In [ ]:
# =========================
# CELL 2: DIRECTORIES & DATA PATHS
# =========================
import os

is_kaggle = Path('/kaggle/input').exists()

if is_kaggle:
    INPUT_DIR = '/kaggle/input/isic-2024-challenge'
    OUTPUT_DIR = '/kaggle/working/optuna_outputs'
else:
    INPUT_DIR = './archive'
    OUTPUT_DIR = './optuna_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_METADATA_PATH = os.path.join(INPUT_DIR, 'train-metadata.csv')
TEST_METADATA_PATH = os.path.join(INPUT_DIR, 'test-metadata.csv')
SAMPLE_SUBMISSION_PATH = os.path.join(INPUT_DIR, 'sample_submission.csv')
TRAIN_HDF5_PATH = os.path.join(INPUT_DIR, 'train-image.hdf5')
TEST_HDF5_PATH = os.path.join(INPUT_DIR, 'test-image.hdf5')

print(f'Train metadata path: {TRAIN_METADATA_PATH}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# =========================
# CELL 3: RUN FEATURE ENGINEERING ONCE
# =========================
import pandas as pd
from tabular_engineering_02 import add_feature_engineering
import config as config_module

print('Pre-loading and running feature engineering to speed up trials...')
train_df = pd.read_csv(TRAIN_METADATA_PATH, low_memory=True)
test_df = pd.read_csv(TEST_METADATA_PATH, low_memory=True)

# Run feature engineering once
train_df, test_df = add_feature_engineering(
    train_df=train_df,
    test_df=test_df,
    group_col='patient_id',
    categorical_cols=config_module.DEFAULT_CATEGORICAL_COLS,
)

# Save pre-engineered datasets
temp_train_path = os.path.join(OUTPUT_DIR, 'pre_engineered_train.csv')
temp_test_path = os.path.join(OUTPUT_DIR, 'pre_engineered_test.csv')
train_df.to_csv(temp_train_path, index=False)
test_df.to_csv(temp_test_path, index=False)

print('Pre-engineered datasets saved successfully.')
print(f'Engineered columns count: {len(train_df.columns)}')

In [ ]:
# =========================
# CELL 4: DEFINE OPTUNA OBJECTIVE FUNCTION
# =========================
import optuna
from config import get_preset_config
from runner import run_mid_fusion_cv

# Build base config mapping to pre-engineered files
base_config = get_preset_config('quick').with_updates(
    train_metadata_path=temp_train_path,
    test_metadata_path=temp_test_path,
    use_feature_engineering=False,   # Already pre-engineered
    n_splits=3,                      # 3-Fold Cross-Validation
    fold_indices=[0, 1, 2],
    modality_mode='tabular_only',
    use_tabm_tabular=True,
    save_checkpoints=False,          # Do not write checkpoint files to disk during trials
    show_predict_progress=False,
    num_workers=0,                   # Enforced DataLoader worker rule
)

def objective(trial):
    # Stage 1: Tabular Optimization Search Space
    n_bins = trial.suggest_int('n_bins', 48, 128, step=16)
    d_embedding = trial.suggest_int('d_embedding', 12, 32, step=4)
    tabm_k = trial.suggest_int('tabm_k', 8, 32, step=8)
    tabm_n_blocks = trial.suggest_int('tabm_n_blocks', 2, 4)
    tabm_d_block = trial.suggest_int('tabm_d_block', 128, 384, step=64)
    tab_dropout = trial.suggest_float('tab_dropout', 0.05, 0.25)
    lr = trial.suggest_float('lr', 1e-4, 8e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    
    # Apply trial updates
    trial_config = base_config.with_updates(
        n_bins=n_bins,
        d_embedding=d_embedding,
        tabm_k=tabm_k,
        tabm_n_blocks=tabm_n_blocks,
        tabm_d_block=tabm_d_block,
        tab_dropout=tab_dropout,
        lr=lr,
        weight_decay=weight_decay,
    )
    
    print(f'\n[Trial {trial.number}] Parameters: {trial.params}')
    try:
        result = run_mid_fusion_cv(trial_config)
        score = result['cv_pauc']
        print(f'[Trial {trial.number}] Complete. OOF pAUC = {score:.6f}')
        return score
    except Exception as e:
        print(f'[Trial {trial.number}] Failed: {e}')
        return float('-inf')

print('Objective function defined.')

In [ ]:
# =========================
# CELL 5: RUN OPTUNA STUDY
# =========================
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, timeout=7200) # Max 2 hours

print('\n' + '=' * 60)
print('OPTUNA OPTIMIZATION COMPLETE')
print('=' * 60)
print(f'Best CV pAUC Score: {study.best_value:.6f}')
print('Best Parameters:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# =========================
# CELL 6: ANALYZE AND SAVE RESULTS
# =========================
trials_df = study.trials_dataframe()
csv_path = os.path.join(OUTPUT_DIR, 'optuna_tabm_trials.csv')
trials_df.to_csv(csv_path, index=False)
print(f'Saved trials results to: {csv_path}')

try:
    import matplotlib.pyplot as plt
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()
    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.show()
except Exception as e:
    print('Could not display visualizations:', e)